In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.utils import power_spectrum
from jaxpm.utils import _initialize_pk
from jaxpm.pm import linear_field, lpt, make_ode_fn, pm_forces
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm import camels

jax.devices()

[CpuDevice(id=0)]

In [3]:
# parts_per_dim = 64
parts_per_dim = 256
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3

CAMELS = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims"
# CODE = "IllustrisTNG"
CODE = "Astrid"
# CODE = "SIMBA"
SIMSET = "CV/CV_0"
# SNAPSHOT = "snapshot_090.hdf5"
SNAPSHOT = "snapshot_014.hdf5"

# mass

In [4]:
FILE = os.path.join(CAMELS, CODE + "_DM", SIMSET, SNAPSHOT)
with h5py.File(FILE, "r") as data:
    n_dm_mass = data["Header"].attrs["MassTable"][1] * 1e10  #  Msun/h
    print(data["Header"].attrs["MassTable"][1] * 1e10)
    # print(data["PartType1/Masses"][:] * 1e10)

FILE = os.path.join(CAMELS, CODE, SIMSET, SNAPSHOT)
with h5py.File(FILE, "r") as data:
    h_dm_mass = data["Header"].attrs["MassTable"][1] * 1e10  #  Msun/h
    h_gas_mass = data["PartType0/Masses"][:] * 1e10

    print(data["Header"].attrs["MassTable"] * 1e10)
    # print(data["PartType1"].keys())
    print(data["PartType1/Masses"][:] * 1e10)

77546567.06246458
[0. 0. 0. 0. 0. 0.]
[64880628. 64880628. 64880628. ... 64880628. 64880628. 64880628.]


In [ ]:
n_dm_mass

In [ ]:
h_dm_mass

In [ ]:
h_gas_mass

In [ ]:
mass_ratio = (h_dm_mass + h_gas_mass) / n_dm_mass
print(mass_ratio)
print(jnp.mean(mass_ratio))

In [ ]:
Omega_c = 0.3 - 0.049
Omega_b = 0.049

In [ ]:
hpm_dm_mass = Omega_c / (Omega_c + Omega_b)
hpm_gas_mass = Omega_b / (Omega_c + Omega_b)

In [ ]:
(jnp.mean(hydro_gas_mass) + hydro_dm_mass) / nbody_dm_mass

In [ ]:
hpm_gas_mass

In [ ]:
hydro_dm_mass/nbody_dm_mass

In [ ]:
Omega_c/(Omega_c + Omega_b)

In [ ]:
jnp.mean(hydro_gas_mass) / hydro_dm_mass

In [ ]:
Omega_b / Omega_c

In [ ]:
nbody_dm_mass

In [ ]:
hpm_dm_mass + hpm_gas_mass